In [41]:
from langgraph.graph import StateGraph,START ,END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver


In [42]:
load_dotenv()

True

In [43]:
GROK_API_KEY = os.getenv("GROK_API_KEY")

llm = ChatOpenAI(
    model="qwen/qwen3.8-27b",
    temperature=0,
    api_key=GROK_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    # Groq free tier caps OUTPUT at 1000 tokens/minute (OTPM). Keep every
    # single response under that so a call isn't rejected before it runs.
    max_tokens=900,
)

In [44]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    
    messages:Annotated[list[BaseMessage],add_messages]

In [45]:
def chat_node(state:ChatState):
    messages=state['messages']
    response=llm.invoke(messages)
    return {'messages':[response]}

In [46]:
# Create the checkpointer only once. Re-running this cell used to replace it
# with a new, empty MemorySaver and wipe every thread's history.
if 'checkpoint' not in globals():
    checkpoint=MemorySaver()

graph=StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

In [47]:
chatbot=graph.compile(checkpointer=checkpoint)

In [48]:
thread_id="1"


initial_state={
    'messages':[HumanMessage(content='What is Object oriented programming')]
}
config={'configurable':{'thread_id':thread_id}}

response=chatbot.invoke(initial_state,config=config)

In [49]:
response['messages'][-1].content

'**Object-Oriented Programming (OOP)** is a programming paradigm (a style or approach to writing code) that organizes software design around **objects** rather than functions and logic.\n\nIn OOP, an **object** is a bundle of data (variables) and the functions (methods) that operate on that data. This approach makes code more modular, reusable, and easier to maintain, especially in large-scale applications.\n\n### Core Concepts of OOP\n\nOOP is built on four fundamental pillars:\n\n1. **Encapsulation**\n   - Bundling data (attributes) and methods (functions) that operate on that data into a single unit (a class).\n   - Hides internal details and implementation from the outside world, exposing only what is necessary.\n   - *Example:* A `BankAccount` class hides the internal balance calculation logic but provides a public method like `deposit(amount)`.\n\n2. **Abstraction**\n   - Hiding complex implementation details and showing only the essential features of an object.\n   - Allows user

In [50]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='What is Object oriented programming', additional_kwargs={}, response_metadata={}, id='3874769b-336d-47ff-bc8a-f0e7c2476a78'), AIMessage(content='**Object-Oriented Programming (OOP)** is a programming paradigm (a style or approach to writing code) that organizes software design around **objects** rather than functions and logic.\n\nIn OOP, an **object** is a bundle of data (variables) and the functions (methods) that operate on that data. This approach makes code more modular, reusable, and easier to maintain, especially in large-scale applications.\n\n### Core Concepts of OOP\n\nOOP is built on four fundamental pillars:\n\n1. **Encapsulation**\n   - Bundling data (attributes) and methods (functions) that operate on that data into a single unit (a class).\n   - Hides internal details and implementation from the outside world, exposing only what is necessary.\n   - *Example:* A `BankAccount` class hides the internal balance calcula

In [51]:
# Type "/thread <id>" to switch conversations without re-running any other cell
while True:
    user_message=input(f'[thread {thread_id}] Type here')
    print('User',user_message)

    if user_message.strip().lower() in ['exit','quit','bye']:
        break
    if user_message.strip().startswith('/thread'):
        thread_id=user_message.split(maxsplit=1)[1].strip()
        print('Switched to thread',thread_id)
        continue
    config={'configurable':{'thread_id':thread_id}}
    response=chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)
    print('Ai:',response['messages'][-1].content)

User exit
